In [5]:
# re-size all the images to this
IMAGE_SIZE = [224, 224]

In [6]:
size = [224, 224] + [3]
size

[224, 224, 3]

In [ ]:
train_path = r'C:\Users\md_alamin\Desktop\skin\dataset\train'
valid_path = r'C:\Users\md_alamin\Desktop\skin\dataset\val'

In [54]:
!nvidia-smi

Fri Apr 10 14:08:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P0             12W /  127W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [55]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.19.0
GPUs: []


In [10]:
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator,load_img
import numpy as np
from glob import glob
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Lambda, Dense, Flatten

In [11]:
vgg16 = VGG16(input_shape=IMAGE_SIZE + [3],  weights='imagenet', include_top=False)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step


In [25]:
for layer in vgg16.layers:
  print(layer.trainable)

False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False


In [24]:
for layer in vgg16.layers:
    layer.trainable = False

In [23]:
for layer in vgg16.layers:
  print(layer.name,layer.trainable)

input_layer False
block1_conv1 False
block1_conv2 False
block1_pool False
block2_conv1 False
block2_conv2 False
block2_pool False
block3_conv1 False
block3_conv2 False
block3_conv3 False
block3_pool False
block4_conv1 False
block4_conv2 False
block4_conv3 False
block4_pool False
block5_conv1 False
block5_conv2 False
block5_conv3 False
block5_pool False


In [26]:
vgg16.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 14,714,688 (56.13 MB)

In [27]:
folders = glob(r'C:\Users\md_alamin\Desktop\skin\dataset\train\*')
folders

['C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\1. Eczema 1677',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\10. Warts Molluscum and other Viral Infections - 2103',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\2. Melanoma 15.75k',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\3. Atopic Dermatitis - 1.25k',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\4. Basal Cell Carcinoma (BCC) 3323',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\5. Melanocytic Nevi (NV) - 7970',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\6. Benign Keratosis-like Lesions (BKL) 2624',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\7. Psoriasis pictures Lichen Planus and related diseases - 2k',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\8. Seborrheic Keratoses and other Benign Tumors - 1.8k',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k']

In [28]:
num_of_class = len(folders)
num_of_class

10

In [29]:
folders = glob(r'C:\Users\md_alamin\Desktop\skin\dataset\train/*')
folders

['C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\1. Eczema 1677',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\10. Warts Molluscum and other Viral Infections - 2103',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\2. Melanoma 15.75k',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\3. Atopic Dermatitis - 1.25k',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\4. Basal Cell Carcinoma (BCC) 3323',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\5. Melanocytic Nevi (NV) - 7970',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\6. Benign Keratosis-like Lesions (BKL) 2624',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\7. Psoriasis pictures Lichen Planus and related diseases - 2k',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\8. Seborrheic Keratoses and other Benign Tumors - 1.8k',
 'C:\\Users\\md_alamin\\Desktop\\skin\\dataset\\train\\9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k']

In [30]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)

In [31]:
train_path = r'C:\Users\md_alamin\Desktop\skin\dataset\train'

train_set = train_datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

Found 19003 images belonging to 10 classes.


In [32]:
val_set = val_datagen.flow_from_directory(
    valid_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

Found 4072 images belonging to 10 classes.


In [34]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_labels = train_set.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(class_labels),
    y=class_labels
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: 1.6200341005967605, 1: 1.2909646739130434, 2: 0.8645586897179254, 3: 2.1618885096700797, 4: 0.816981943250215, 5: 0.340616597956623, 6: 1.3060481099656358, 7: 1.3214881780250347, 8: 1.4708204334365325, 9: 1.5955499580184718}


In [35]:
model = Sequential()
model.add(vgg16)
model.add(Flatten())
model.add(Dense(256,activation='relu'))
model.add(Dense(num_of_class,activation='softmax'))

In [36]:
# view the structure of the model
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,140,042 (80.64 MB)

 Trainable params: 6,425,354 (24.51 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [38]:
model.compile(
  loss='categorical_crossentropy',
  optimizer='adam',
  metrics=['accuracy']
)

In [39]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [40]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)

test_datagen = ImageDataGenerator(rescale = 1./255)

In [42]:
import os



for folder in os.listdir(train_path):
    folder_path = os.path.join(train_path, folder)

    if os.path.isdir(folder_path):
        count = len(os.listdir(folder_path))
        print(f"{folder} → {count} images")

1. Eczema 1677 → 1173 images
10. Warts Molluscum and other Viral Infections - 2103 → 1472 images
2. Melanoma 15.75k → 2198 images
3. Atopic Dermatitis - 1.25k → 879 images
4. Basal Cell Carcinoma (BCC) 3323 → 2326 images
5. Melanocytic Nevi (NV) - 7970 → 5579 images
6. Benign Keratosis-like Lesions (BKL) 2624 → 1455 images
7. Psoriasis pictures Lichen Planus and related diseases - 2k → 1438 images
8. Seborrheic Keratoses and other Benign Tumors - 1.8k → 1292 images
9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k → 1191 images


In [43]:
training_set = train_datagen.flow_from_directory(train_path,
                                                 target_size = (224, 224),
                                                 batch_size = 32,
                                                 class_mode = 'categorical')

Found 19003 images belonging to 10 classes.


In [44]:
test_set = test_datagen.flow_from_directory(valid_path,
                                            target_size = (224, 224),
                                            batch_size = 32,
                                            class_mode = 'categorical')

Found 4072 images belonging to 10 classes.


In [46]:
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available: 0


In [48]:
!nvidia-smi

Fri Apr 10 13:28:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P0             12W /  127W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [50]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

[]


In [ ]:
r = model.fit(
  training_set,
  validation_data=test_set,
  epochs=10,
  steps_per_epoch=len(training_set),
  validation_steps=len(test_set)
)  

In [ ]:
import matplotlib.pyplot as plt
# plot the loss
plt.plot(r.history['loss'], label='train loss')
plt.plot(r.history['val_loss'], label='val loss')
plt.legend()
plt.show()
plt.savefig('LossVal_loss')

In [ ]:
# plot the accuracy
plt.plot(r.history['accuracy'], label='train acc')
plt.plot(r.history['val_accuracy'], label='val acc')
plt.legend()
plt.show()
plt.savefig('AccVal_acc')

In [ ]:
model.evaluate(test_set)

In [ ]:
from tensorflow.keras.models import load_model

model.save('model_vgg16.h5')

In [ ]:
model = load_model("model_vgg16.h5")

In [ ]:
from tensorflow.keras.preprocessing import image

In [ ]:
img = "/content/Bones-data/test/Oblique fracture/43390tn_jpg.rf.e82c12a328a56cc66d5d828e638324be.jpg"

In [ ]:
img=image.load_img(img,target_size=(224,224))

In [ ]:
img

In [ ]:
x=image.img_to_array(img)
x

In [ ]:
Z = plt.imread('/content/Bones-data/test/Oblique fracture/43390tn_jpg.rf.e82c12a328a56cc66d5d828e638324be.jpg')
plt.imshow(Z)

In [ ]:
x.shape

In [ ]:
x=x/255

In [ ]:
from keras.applications.vgg16 import preprocess_input
import numpy as np
x=np.expand_dims(x,axis=0)
img_data=preprocess_input(x)

In [ ]:
img_data

In [ ]:
result = np.argmax(output, axis=1)
result

In [ ]:
if result[0] == 0:
    prediction = 'Oblique fracture'
    print(prediction)
else:
    prediction = 'Spiral Fracture'
    print(prediction)